# 🚀 CASA Paper-Ready Benchmark Runner on Kaggle GPU

This notebook provides the complete reproducible pipeline to run **CASA (Coalition-Aware Sparse Adversarial Attack)** and competitor baselines (**SPGD, Sigma-Zero, Sparse-RS, sAA**) on the **full 10,000 CIFAR-10 test set**.

### Recommended Kaggle Settings:
- **Accelerator:** GPU T4 x2 or GPU P100 (Settings menu on the right panel)
- **Persistence:** Files only (or Variables and Files)
- **Internet:** Enabled (required to clone repository & download HuggingFace CIFAR-10 test set)

---

In [ ]:
# Step 0: Clone Repository & Setup Working Directory
import os, sys, subprocess

REPO_URL = "https://github.com/nxc1802/AA_2.git"
REPO_NAME = "AA_2"

# 1. Clone repository if not already cloned
if not os.path.exists("pyproject.toml"):
    if not os.path.exists(REPO_NAME):
        print(f"Cloning {REPO_URL} into {REPO_NAME}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"Repository {REPO_NAME} already exists. Pulling latest updates...")
        subprocess.run(["git", "-C", REPO_NAME, "pull"], check=False)
    os.chdir(REPO_NAME)

print("Current Working Directory:", os.getcwd())
assert os.path.exists("pyproject.toml"), "Error: pyproject.toml not found! Check directory structure."

# 2. Add src/ to sys.path for direct imports
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Environment path configured successfully.")

In [ ]:
# Step 1: GPU Diagnostics
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Step 2: Install dependencies & AA package
!pip install -q datasets huggingface_hub lpips pyyaml
!pip install -e . --quiet
print("Environment setup complete.")

In [ ]:
# Step 3: Checkpoint Verification (SHA256 Gate)
import os, hashlib, shutil
from aa.models import find_existing_checkpoint

ckpt_path = "result/saved_models/resnet18_cifar10_best.pth"
expected_sha = "378eb005089d3942a3f237aeb08a927aa3dfbe41535c364891468b33c87d2172"

os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
if not os.path.exists(ckpt_path):
    print("Downloading official verified checkpoint...")
    resolved = find_existing_checkpoint(ckpt_path)
    if resolved and os.path.isfile(resolved) and resolved != ckpt_path:
        shutil.copy2(resolved, ckpt_path)

with open(ckpt_path, "rb") as f:
    actual_sha = hashlib.sha256(f.read()).hexdigest()

assert actual_sha == expected_sha, f"Checksum error! Expected {expected_sha}, got {actual_sha}"
print(f"✅ Checkpoint verified: {ckpt_path}")
print(f"   SHA256: {actual_sha}")

In [ ]:
# Step 4: Run Smoke Sanity Check (20 samples)
!python scripts/attack_benchmark.py --config configs/smoke.yaml --strict --output result/smoke_results.json

In [ ]:
# Step 5: [STAGE 1] Run CASA Official 10k Benchmark on CIFAR-10
# Evaluates K in {1, 2, 4, 8, 16, 32, 64} on all 10,000 test images.
!python scripts/run_casa_benchmark.py \
    --samples 10000 \
    --batch-size 16 \
    --k-values 1 2 4 8 16 32 64 \
    --output result/casa_10000_results.json

In [ ]:
# Step 6: [STAGE 2] Run Strong Baselines (SPGD & Sigma-Zero) on 10k samples
!python scripts/attack_benchmark.py \
    --config configs/paper_cifar10.yaml \
    --attacks spgd,sigma_zero \
    --output result/baselines_spgd_sigmazero_10k.json

In [ ]:
# Step 7: [STAGE 3] Run Component Ablation Study (1,000 samples, K=1, 4, 16)
!python scripts/run_ablation.py \
    --samples 1000 \
    --k-values 1 4 16 \
    --output result/ablation_results.json \
    --report-md docs/ablation_study.md

In [ ]:
# Step 8: [STAGE 4] Failure Case Diagnostic Analysis
!python scripts/analyze_failures.py \
    --samples 1000 \
    --k-values 1 2 4 \
    --max-failures 30 \
    --output-json result/failure_analysis.json \
    --output-md docs/failure_analysis.md

In [ ]:
# Step 9: [STAGE 5] Generate High-Resolution Paper Figures (300 DPI)
!python scripts/plot_paper_figures.py --output-dir result/figures

# Display figures inline
from IPython.display import Image, display
import glob
for fig_path in sorted(glob.glob("result/figures/*.png")):
    print(f"\n--- Displaying {fig_path} ---")
    display(Image(filename=fig_path, width=650))

In [ ]:
# Step 10: Package All Artifacts for Download
!tar -czvf paper_artifacts.tar.gz result/ docs/ configs/
!cp paper_artifacts.tar.gz /kaggle/working/ 2>/dev/null || true
print("\n✅ All results packaged into paper_artifacts.tar.gz. Download from Kaggle Output panel.")